In [ ]:
import sympy as sym
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import numpy as np
import pandas as pd

# Load in the required datasets
gld_mass_balance_filename = 'Datasets/ice-sheet-mass-balance.csv' #relative filepath
gld_mass_balance_csv = pd.read_csv(gld_mass_balance_filename)

gld_mass_balance_csv["Day"] = pd.to_datetime(gld_mass_balance_csv["Day"])

plt.figure(figsize=(10,5))
plt.plot(gld_mass_balance_csv["Day"], gld_mass_balance_csv["Seasonal variation"])
plt.xlabel("Year"); plt.ylabel("Greenland Ice Sheet Mass Balance Change")
plt.show()

# Calibration value: 2001 thickness of around 1600m (Thomas et al, 2001)
calb_year = pd.to_datetime('2017-01-01')
calb_avg_thickness = 1673 #m

# Let the ice cap mass be directly proportional to the height of the ice cap 
# Assumption: Volume = w * l * h, and its a prism with constant cross-sectional area
# So let V be directly prop. to h. 
# V = m * density, so assume constant density also.

# Recalibrate this plot to be height change over time. 
date_idx = gld_mass_balance_csv["Day"].sub(calb_year).abs().idxmin()
calb_mass = gld_mass_balance_csv.loc[date_idx, "Seasonal variation"]

calb_offset = calb_avg_thickness - calb_mass

gld_mass_balance_csv["height calibrated"] = gld_mass_balance_csv["Seasonal variation"] + calb_offset

plt.figure(figsize=(10,5))
plt.plot(gld_mass_balance_csv["Day"], gld_mass_balance_csv["height calibrated"])
plt.xlabel("Year"); plt.ylabel("Greenland Ice Sheet Height (Calibrated)")
plt.show()



In [ ]:


dT, h = sym.symbols('dT, h')

# T0 = -1.5 #degC
# Tm = 0 #degC
# P  = 0.3 #m (up to 1.5m apparently)


# F  = 10 # m/year? 
# r = 1e4 # dimensionless?

######### Attempt to make more informed guess on the values! 

h0 = 3000
Tm = 0 #fairly certain of this fact
T0 = -10 #degC, roughly!
P = 30 #m/year # Glacier - Greenland, Ice Sheet, Melting | Britannica.... REF

# Thus we must enforce that P - F*h0 must be > 0. Since we have a good guess of P, 
# F < P/h0 

F = 0.8 * (P/h0)
print(f"F = {F}")
print(f"P/h0 = {P/h0}")
print(f"P - F*h0 = {(P - F*h0)}")

r = (h0 * (P - (F*h0))) / ((T0 - Tm)**2)

print(f"r = {r}")


def delta_T(t): 
    return 5 + 0.01*t #linear warming of 0.1degC per year 


def dh_dt(t, h):
    dT = delta_T(t)

    h_safe = np.maximum(h, 1.0) # enforcing a muin h so that the solver doesnt break with high gradients

    dhdt = P - (r * (T0 + dT - Tm)**2)/h_safe - F*h_safe
    dhdt = np.where((h <= 0) & (dhdt < 0), 0, dhdt)
    return dhdt


h_vals = np.linspace(50, 4000, 500)  # avoid h=0 to prevent division
plt.figure()
plt.plot(h_vals, dh_dt(0, h_vals))
plt.axhline(0, linestyle='--')
plt.xlabel('h (m)')
plt.ylabel('dh/dt (m/year)')
plt.grid()
plt.show()


# Compute tipping point 

dh_dt_symb =  P - (( r *(T0 + dT - Tm) **2 )/h) - (F*h)

func = sym.simplify(dh_dt_symb * h)
discr = sym.discriminant(func, h) 
print(f"Distriminant: {discr}")

tipping_points = sym.solve(discr, dT)
print("Tipping points:", tipping_points)





In [ ]:

h_vals = np.linspace(50, 2500, 500)  # reasonable range
deltaTs = [0, 5, 10, 25]  # example temperature increases

plt.figure(figsize=(6,4))
for dT in deltaTs:
    dhdt_vals = P - (r * (T0 + dT - Tm)**2)/h_vals - F*h_vals
    plt.plot(h_vals, dhdt_vals, label=f'ΔT={dT}°C')
plt.axhline(0, linestyle='--', color='k')
plt.xlabel('Ice sheet height h (m)')
plt.ylabel('dh/dt (m/year)')
plt.title('dh/dt vs h for different ΔT')
plt.grid()
plt.legend()
plt.show()

t_upper = 10000
t_span = [0, t_upper]
t_eval = np.linspace(0, t_upper, 500)
h0_1 = 100
h0_2 = 1500

sol1 = solve_ivp(dh_dt, t_span, [h0_1], t_eval=t_eval)
sol2 = solve_ivp(dh_dt, t_span, [h0_2], t_eval=t_eval)

plt.figure(figsize=(6,4))
plt.plot(sol1.t, sol1.y[0], label=f'h0={h0_1} m')
plt.plot(sol2.t, sol2.y[0], label=f'h0={h0_2} m')
plt.xlabel('Time (years)')
plt.ylabel('Ice sheet height h(t) (m)')
plt.title('Ice sheet evolution over time')
plt.grid()
plt.legend()
plt.show()

In [ ]:
t_start = 0
t_end   = 200
t = np.linspace(t_start, t_end, 10)

# h0_1 = 1.5
# h0_2 = 2.5

h0_1 = tipping_points[0] *0.9
h0_2 = tipping_points[1] *1.1
sol1 = solve_ivp(dh_dt, [t_start, t_end], [h0_1], t_eval=t)
sol2 = solve_ivp(dh_dt, [t_start, t_end], [h0_2], t_eval=t)

plt.plot(sol1.t, sol1.y[0], label=f'h0={h0_1} m')
plt.plot(sol2.t, sol2.y[0], label=f'h0={h0_2} m')
plt.plot(sol1.t, delta_T(sol1.t), label=r'$\delta T(t)$')
# plt.hlines(6.25, 0, 200, 'r', '--')
# plt.vlines(125, 0, 7, 'r', '--')

plt.xlabel('Time (years)')
plt.ylabel('Ice sheet height h(t) (m)')

plt.grid()
plt.legend()
plt.show()


In [ ]:
ax = plt.figure().add_subplot(111)

h_val = 1000

hv = np.linspace(0, 2000, 100)
ax.plot(hv, dh_dt(hv, h))

ax.set_xlabel(r'$h$')
ax.set_ylabel(r'$f(x, h)$', rotation=0)
ax.hlines(0, -1.5, 1.5, linestyles='dashed')

for eq in t_cond:
    ax.plot(eq, 0, 'ro')



dh_dt = P - (( r *(T0 + dT - Tm) **2 )/h) - (F*h)

def dh_dt(h, dT):
    return P - (r * (T0 + dT - Tm)**2)/h - F*h


h_vals = np.linspace(1, 2000, 200) 
dT_val = 0.5

plt.figure()
plt.plot(h_vals, dh_dt(h_vals, dT_val))
plt.axhline(0, linestyle='dashed')
plt.xlabel('h')
plt.ylabel('dh/dt')
plt.grid()
plt.show()


func = sym.simplify(dh_dt * h)

discr = sym.discriminant(func, h) 
#this is just expanded form of:
# P^2 -4Fr(T0 + dT -Tm)^2

# Find real roots: 
rl_rts = sym.real_root(discr)

t_cond = sym.solve(discr, dT)
print(f"Equilibira: {t_cond}")


fig = plt.figure()
ax1 = fig.add_subplot(121)
ax2 = fig.add_subplot(122)

# delta_t = 0.1